# RC/SPA тесты (1d U17, 1d U12, 4h U12)

### Setup + Config

In [ ]:
from __future__ import annotations

import importlib.util
import logging
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src.rc_spa import run_rc_spa_returns, run_rc_spa_utility
from src import stats_tests as st_module

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("rc_spa_tests_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

RESULTS_DIR = PROJECT_ROOT / "results" / "rc_spa_tests"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "base_1d_returns": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "returns_oos.parquet",
    "base_1d_bench": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "bench_returns_oos.parquet",
    "onchain_1d_returns": PROJECT_ROOT / "results" / "runner_onchain" / "returns_oos.parquet",
    "base_4h_returns": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "returns_oos.parquet",
    "base_4h_bench": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "bench_returns_oos.parquet",
}

SYMBOLS_1D = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]
SYMBOLS_4H = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]
COSTS = ["Low", "Base", "High"]

RC_PARAMS = {
    "reps": 2000,
    "seed": 42,
    "studentize": True,
    "alpha": 0.05,
}
LAMBDAS = [0.5, 1.0, 2.0]
BLOCK_SIZE = {"1d": 10, "4h": 60}

for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name} -> {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("arch installed in current kernel:", bool(importlib.util.find_spec("arch")))
print("inputs:")
for name, path in INPUTS.items():
    print(f"  - {name}: {path}")


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\rc_spa_tests
arch installed in current kernel: True
inputs:
  - base_1d_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\1d\returns_oos.parquet
  - base_1d_bench: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\1d\bench_returns_oos.parquet
  - onchain_1d_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain\returns_oos.parquet
  - base_4h_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\4h\returns_oos.parquet
  - base_4h_bench: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\4h\bench_returns_oos.parquet


In [ ]:
def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date-like column found. Columns: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def assert_unique_keys(df: pd.DataFrame, *, name: str) -> None:
    key_cols = ["date", "symbol", "cost", "strategy_id"]
    dups = int(df.duplicated(subset=key_cols).sum())
    if dups > 0:
        raise RuntimeError(f"{name}: found duplicate rows by {key_cols}: {dups}")


def coverage_snapshot(df: pd.DataFrame, *, name: str) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "dataset": [name],
            "rows": [len(df)],
            "symbols": [df["symbol"].astype(str).nunique()],
            "costs": [df["cost"].astype(str).nunique()],
            "strategies": [df["strategy_id"].astype(str).nunique()],
            "min_date": [pd.to_datetime(df["date"]).min()],
            "max_date": [pd.to_datetime(df["date"]).max()],
        }
    )
    return out


base_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["base_1d_returns"]))
base_1d_bench = ensure_date_col(pd.read_parquet(INPUTS["base_1d_bench"]))
onchain_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["onchain_1d_returns"]))
base_4h_returns = ensure_date_col(pd.read_parquet(INPUTS["base_4h_returns"]))
base_4h_bench = ensure_date_col(pd.read_parquet(INPUTS["base_4h_bench"]))

assert_unique_keys(base_1d_returns, name="base_1d_returns")
assert_unique_keys(base_1d_bench, name="base_1d_bench")
assert_unique_keys(onchain_1d_returns, name="onchain_1d_returns")
assert_unique_keys(base_4h_returns, name="base_4h_returns")
assert_unique_keys(base_4h_bench, name="base_4h_bench")

snapshots = pd.concat(
    [
        coverage_snapshot(base_1d_returns, name="base_1d_returns"),
        coverage_snapshot(base_1d_bench, name="base_1d_bench"),
        coverage_snapshot(onchain_1d_returns, name="onchain_1d_returns"),
        coverage_snapshot(base_4h_returns, name="base_4h_returns"),
        coverage_snapshot(base_4h_bench, name="base_4h_bench"),
    ],
    ignore_index=True,
)

display(snapshots)

base_1d_nsid = base_1d_returns["strategy_id"].astype(str).nunique()
onchain_1d_nsid = onchain_1d_returns["strategy_id"].astype(str).nunique()
base_4h_nsid = base_4h_returns["strategy_id"].astype(str).nunique()

print("base_1d strategy count:", base_1d_nsid)
print("onchain_1d strategy count:", onchain_1d_nsid)
print("base_4h strategy count:", base_4h_nsid)

if base_1d_nsid != 12:
    raise AssertionError(f"Expected 12 base strategies on 1d, got {base_1d_nsid}")
if onchain_1d_nsid != 5:
    raise AssertionError(f"Expected 5 onchain strategies on 1d, got {onchain_1d_nsid}")
if base_4h_nsid != 12:
    raise AssertionError(f"Expected 12 base strategies on 4h, got {base_4h_nsid}")

print("Sanity checks passed.")


,dataset,rows,symbols,costs,strategies,min_date,max_date
0,base_1d_returns,315504,5,3,12,2020-09-16 00:00:00,2026-03-15
1,base_1d_bench,26292,5,3,1,2020-09-16 00:00:00,2026-03-15
2,onchain_1d_returns,104070,4,3,5,2020-09-16 00:00:00,2026-03-15
3,base_4h_returns,1881792,5,3,12,2020-08-22 04:00:00,2026-02-22
4,base_4h_bench,156816,5,3,1,2020-08-22 04:00:00,2026-02-22


base_1d strategy count: 12
onchain_1d strategy count: 5
base_4h strategy count: 12
Sanity checks passed.


### 1d U17 (base + onchain)

In [ ]:
returns_u17_1d = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
assert_unique_keys(returns_u17_1d, name="returns_u17_1d")
bench_u17_1d = base_1d_bench.copy()

rc_returns_1d_u17 = run_rc_spa_returns(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u17_1d,
    bench_long=bench_u17_1d,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_1d_u17["timeframe"] = "1d"
rc_returns_1d_u17["scenario"] = "u17"

rc_utility_1d_u17 = run_rc_spa_utility(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u17_1d,
    bench_long=bench_u17_1d,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_1d_u17["timeframe"] = "1d"
rc_utility_1d_u17["scenario"] = "u17"

safe_write_parquet(rc_returns_1d_u17, RESULTS_DIR / "rc_spa_returns_1d_u17.parquet")
safe_write_parquet(rc_utility_1d_u17, RESULTS_DIR / "rc_spa_utility_1d_u17.parquet")

print("rc_returns_1d_u17:", rc_returns_1d_u17.shape)
print("rc_utility_1d_u17:", rc_utility_1d_u17.shape)
display(rc_returns_1d_u17.head(10))
display(rc_utility_1d_u17.head(10))


2026-04-07 14:43:36,168 | INFO | RC/SPA universe: BTCUSDT | Low | T=2007 | models=17
2026-04-07 14:43:48,469 | INFO | RC/SPA universe: BTCUSDT | Base | T=2007 | models=17
2026-04-07 14:43:53,753 | INFO | RC/SPA universe: BTCUSDT | High | T=2007 | models=17
2026-04-07 14:43:59,186 | INFO | RC/SPA universe: ETHUSDT | Low | T=2007 | models=17
2026-04-07 14:44:04,386 | INFO | RC/SPA universe: ETHUSDT | Base | T=2007 | models=17
2026-04-07 14:44:09,675 | INFO | RC/SPA universe: ETHUSDT | High | T=2007 | models=17
2026-04-07 14:44:14,923 | INFO | RC/SPA universe: BNBUSDT | Low | T=1826 | models=12
2026-04-07 14:44:19,602 | INFO | RC/SPA universe: BNBUSDT | Base | T=1826 | models=12
2026-04-07 14:44:24,401 | INFO | RC/SPA universe: BNBUSDT | High | T=1826 | models=12
2026-04-07 14:44:29,083 | INFO | RC/SPA universe: XRPUSDT | Low | T=1644 | models=17
2026-04-07 14:44:33,678 | INFO | RC/SPA universe: XRPUSDT | Base | T=1644 | models=17
2026-04-07 14:44:38,012 | INFO | RC/SPA universe: XRPUSDT 

rc_returns_1d_u17: (15, 16)
rc_utility_1d_u17: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,2007,17,0.873063,-0.000066,T2:EMA_Crossover,0.6020,0.8460,0.8740,2000.0,10.0,1.0,0.0,1d,u17
1,BTCUSDT,Base,2007,17,0.873563,-0.000072,T2:EMA_Crossover,0.6185,0.8470,0.8760,2000.0,10.0,1.0,0.0,1d,u17
2,BTCUSDT,High,2007,17,0.890055,-0.000111,T2:EMA_Crossover,0.6380,0.8785,0.8960,2000.0,10.0,1.0,0.0,1d,u17
3,ETHUSDT,Low,2007,17,0.734133,0.000221,T1:SMA_Crossover,0.5125,0.7240,0.7450,2000.0,10.0,1.0,0.0,1d,u17
4,ETHUSDT,Base,2007,17,0.735632,0.000216,T1:SMA_Crossover,0.5155,0.7275,0.7475,2000.0,10.0,1.0,0.0,1d,u17
5,ETHUSDT,High,2007,17,0.737631,0.000206,T1:SMA_Crossover,0.5205,0.7340,0.7545,2000.0,10.0,1.0,0.0,1d,u17
6,BNBUSDT,Low,1826,12,0.962519,-0.000226,T2:EMA_Crossover,0.6150,0.9335,0.9495,2000.0,10.0,1.0,0.0,1d,u17
7,BNBUSDT,Base,1826,12,0.963018,-0.000232,T2:EMA_Crossover,0.6125,0.9355,0.9505,2000.0,10.0,1.0,0.0,1d,u17
8,BNBUSDT,High,1826,12,0.936032,-0.000141,T1:SMA_Crossover,0.5775,0.9115,0.9335,2000.0,10.0,1.0,0.0,1d,u17
9,XRPUSDT,Low,1644,17,0.838581,0.000097,R2:Bollinger_MR,0.6285,0.8335,0.8490,2000.0,10.0,1.0,0.0,1d,u17


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,2007,17,0.026987,0.001762,S1OC:MAFilter_RSI_MR_Onchain,0.0355,0.0355,0.0355,2000.0,10.0,1.0,2.0,1d,u17
1,BTCUSDT,Low,1.0,2007,17,0.000500,0.004792,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
2,BTCUSDT,Low,2.0,2007,17,0.000500,0.010851,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
3,BTCUSDT,Base,0.5,2007,17,0.028486,0.001759,S1OC:MAFilter_RSI_MR_Onchain,0.0355,0.0355,0.0355,2000.0,10.0,1.0,2.0,1d,u17
4,BTCUSDT,Base,1.0,2007,17,0.000500,0.004788,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
5,BTCUSDT,Base,2.0,2007,17,0.000500,0.010844,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
6,BTCUSDT,High,0.5,2007,17,0.032484,0.001754,S1OC:MAFilter_RSI_MR_Onchain,0.0380,0.0380,0.0380,2000.0,10.0,1.0,2.0,1d,u17
7,BTCUSDT,High,1.0,2007,17,0.000500,0.004779,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
8,BTCUSDT,High,2.0,2007,17,0.000500,0.010829,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
9,ETHUSDT,Low,0.5,2007,17,0.021489,0.002370,S2:MA200Filter_Bollinger_MR,0.0295,0.0295,0.0295,2000.0,10.0,1.0,5.0,1d,u17


### 1d U12 (base only)

In [ ]:
returns_u12_1d = base_1d_returns.copy()
bench_u12_1d = base_1d_bench.copy()

rc_returns_1d_u12 = run_rc_spa_returns(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u12_1d,
    bench_long=bench_u12_1d,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_1d_u12["timeframe"] = "1d"
rc_returns_1d_u12["scenario"] = "u12"

rc_utility_1d_u12 = run_rc_spa_utility(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u12_1d,
    bench_long=bench_u12_1d,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_1d_u12["timeframe"] = "1d"
rc_utility_1d_u12["scenario"] = "u12"

safe_write_parquet(rc_returns_1d_u12, RESULTS_DIR / "rc_spa_returns_1d_u12.parquet")
safe_write_parquet(rc_utility_1d_u12, RESULTS_DIR / "rc_spa_utility_1d_u12.parquet")

print("rc_returns_1d_u12:", rc_returns_1d_u12.shape)
print("rc_utility_1d_u12:", rc_utility_1d_u12.shape)
display(rc_returns_1d_u12.head(10))
display(rc_utility_1d_u12.head(10))


2026-04-07 14:48:20,969 | INFO | RC/SPA universe: BTCUSDT | Low | T=2007 | models=12
2026-04-07 14:48:26,285 | INFO | RC/SPA universe: BTCUSDT | Base | T=2007 | models=12
2026-04-07 14:48:31,725 | INFO | RC/SPA universe: BTCUSDT | High | T=2007 | models=12
2026-04-07 14:48:36,887 | INFO | RC/SPA universe: ETHUSDT | Low | T=2007 | models=12
2026-04-07 14:48:42,080 | INFO | RC/SPA universe: ETHUSDT | Base | T=2007 | models=12
2026-04-07 14:48:47,224 | INFO | RC/SPA universe: ETHUSDT | High | T=2007 | models=12
2026-04-07 14:48:52,442 | INFO | RC/SPA universe: BNBUSDT | Low | T=1826 | models=12
2026-04-07 14:48:57,283 | INFO | RC/SPA universe: BNBUSDT | Base | T=1826 | models=12
2026-04-07 14:49:01,992 | INFO | RC/SPA universe: BNBUSDT | High | T=1826 | models=12
2026-04-07 14:49:06,747 | INFO | RC/SPA universe: XRPUSDT | Low | T=1644 | models=12
2026-04-07 14:49:10,958 | INFO | RC/SPA universe: XRPUSDT | Base | T=1644 | models=12
2026-04-07 14:49:15,527 | INFO | RC/SPA universe: XRPUSDT 

rc_returns_1d_u12: (15, 16)
rc_utility_1d_u12: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,2007,12,0.871564,-0.000066,T2:EMA_Crossover,0.6020,0.8325,0.8720,2000.0,10.0,1.0,0.0,1d,u12
1,BTCUSDT,Base,2007,12,0.871064,-0.000072,T2:EMA_Crossover,0.6185,0.8335,0.8740,2000.0,10.0,1.0,0.0,1d,u12
2,BTCUSDT,High,2007,12,0.886557,-0.000111,T2:EMA_Crossover,0.6380,0.8755,0.8935,2000.0,10.0,1.0,0.0,1d,u12
3,ETHUSDT,Low,2007,12,0.728136,0.000221,T1:SMA_Crossover,0.5035,0.7105,0.7400,2000.0,10.0,1.0,0.0,1d,u12
4,ETHUSDT,Base,2007,12,0.729635,0.000216,T1:SMA_Crossover,0.5065,0.7140,0.7425,2000.0,10.0,1.0,0.0,1d,u12
5,ETHUSDT,High,2007,12,0.731634,0.000206,T1:SMA_Crossover,0.5120,0.7205,0.7485,2000.0,10.0,1.0,0.0,1d,u12
6,BNBUSDT,Low,1826,12,0.962519,-0.000226,T2:EMA_Crossover,0.6150,0.9335,0.9495,2000.0,10.0,1.0,0.0,1d,u12
7,BNBUSDT,Base,1826,12,0.963018,-0.000232,T2:EMA_Crossover,0.6125,0.9355,0.9505,2000.0,10.0,1.0,0.0,1d,u12
8,BNBUSDT,High,1826,12,0.936032,-0.000141,T1:SMA_Crossover,0.5775,0.9115,0.9335,2000.0,10.0,1.0,0.0,1d,u12
9,XRPUSDT,Low,1644,12,0.834083,0.000097,R2:Bollinger_MR,0.6070,0.8305,0.8465,2000.0,10.0,1.0,0.0,1d,u12


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,2007,12,0.029485,0.001658,S3:Breakout_Confirm_MA,0.0450,0.0450,0.0450,2000.0,10.0,1.0,1.0,1d,u12
1,BTCUSDT,Low,1.0,2007,12,0.000500,0.004477,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
2,BTCUSDT,Low,2.0,2007,12,0.000500,0.010274,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
3,BTCUSDT,Base,0.5,2007,12,0.031484,0.001655,S3:Breakout_Confirm_MA,0.0450,0.0450,0.0450,2000.0,10.0,1.0,1.0,1d,u12
4,BTCUSDT,Base,1.0,2007,12,0.000500,0.004474,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
5,BTCUSDT,Base,2.0,2007,12,0.000500,0.010270,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
6,BTCUSDT,High,0.5,2007,12,0.036482,0.001650,S3:Breakout_Confirm_MA,0.0475,0.0475,0.0475,2000.0,10.0,1.0,1.0,1d,u12
7,BTCUSDT,High,1.0,2007,12,0.000500,0.004468,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
8,BTCUSDT,High,2.0,2007,12,0.000500,0.010263,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
9,ETHUSDT,Low,0.5,2007,12,0.017491,0.002370,S2:MA200Filter_Bollinger_MR,0.0230,0.0230,0.0230,2000.0,10.0,1.0,2.0,1d,u12


### 4h U12 (base only)

In [ ]:
returns_u12_4h = base_4h_returns.copy()
bench_u12_4h = base_4h_bench.copy()

rc_returns_4h_u12 = run_rc_spa_returns(
    symbols=SYMBOLS_4H,
    costs=COSTS,
    returns_oos_long=returns_u12_4h,
    bench_long=bench_u12_4h,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["4h"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_4h_u12["timeframe"] = "4h"
rc_returns_4h_u12["scenario"] = "u12"

rc_utility_4h_u12 = run_rc_spa_utility(
    symbols=SYMBOLS_4H,
    costs=COSTS,
    returns_oos_long=returns_u12_4h,
    bench_long=bench_u12_4h,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["4h"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_4h_u12["timeframe"] = "4h"
rc_utility_4h_u12["scenario"] = "u12"

safe_write_parquet(rc_returns_4h_u12, RESULTS_DIR / "rc_spa_returns_4h_u12.parquet")
safe_write_parquet(rc_utility_4h_u12, RESULTS_DIR / "rc_spa_utility_4h_u12.parquet")

print("rc_returns_4h_u12:", rc_returns_4h_u12.shape)
print("rc_utility_4h_u12:", rc_utility_4h_u12.shape)
display(rc_returns_4h_u12.head(10))
display(rc_utility_4h_u12.head(10))


2026-04-07 14:53:01,002 | INFO | RC/SPA universe: BTCUSDT | Low | T=12060 | models=12
2026-04-07 14:53:30,080 | INFO | RC/SPA universe: BTCUSDT | Base | T=12060 | models=12
2026-04-07 14:53:58,723 | INFO | RC/SPA universe: BTCUSDT | High | T=12060 | models=12
2026-04-07 14:54:27,327 | INFO | RC/SPA universe: ETHUSDT | Low | T=12060 | models=12
2026-04-07 14:54:55,964 | INFO | RC/SPA universe: ETHUSDT | Base | T=12060 | models=12
2026-04-07 14:55:24,488 | INFO | RC/SPA universe: ETHUSDT | High | T=12060 | models=12
2026-04-07 14:55:52,493 | INFO | RC/SPA universe: BNBUSDT | Low | T=10956 | models=12
2026-04-07 14:56:17,776 | INFO | RC/SPA universe: BNBUSDT | Base | T=10956 | models=12
2026-04-07 14:56:43,263 | INFO | RC/SPA universe: BNBUSDT | High | T=10956 | models=12
2026-04-07 14:57:08,255 | INFO | RC/SPA universe: XRPUSDT | Low | T=9516 | models=12
2026-04-07 14:57:29,811 | INFO | RC/SPA universe: XRPUSDT | Base | T=9516 | models=12
2026-04-07 14:57:50,812 | INFO | RC/SPA universe:

rc_returns_4h_u12: (15, 16)
rc_utility_4h_u12: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,12060,12,0.888556,-0.000010,S5:Ensemble_3Signals,0.6495,0.8275,0.8795,2000.0,60.0,1.0,0.0,4h,u12
1,BTCUSDT,Base,12060,12,0.894053,-0.000011,S5:Ensemble_3Signals,0.6505,0.8265,0.8865,2000.0,60.0,1.0,0.0,4h,u12
2,BTCUSDT,High,12060,12,0.901549,-0.000012,S5:Ensemble_3Signals,0.6330,0.8395,0.8945,2000.0,60.0,1.0,0.0,4h,u12
3,ETHUSDT,Low,12060,12,0.852074,0.000008,S5:Ensemble_3Signals,0.6905,0.8270,0.8340,2000.0,60.0,1.0,0.0,4h,u12
4,ETHUSDT,Base,12060,12,0.853073,0.000006,S5:Ensemble_3Signals,0.6870,0.8305,0.8360,2000.0,60.0,1.0,0.0,4h,u12
5,ETHUSDT,High,12060,12,0.855072,0.000004,S5:Ensemble_3Signals,0.6480,0.8360,0.8405,2000.0,60.0,1.0,0.0,4h,u12
6,BNBUSDT,Low,10956,12,0.991504,-0.000071,T2:EMA_Crossover,0.6345,0.9890,0.9895,2000.0,60.0,1.0,0.0,4h,u12
7,BNBUSDT,Base,10956,12,0.995002,-0.000075,T2:EMA_Crossover,0.6400,0.9910,0.9915,2000.0,60.0,1.0,0.0,4h,u12
8,BNBUSDT,High,10956,12,0.995502,-0.000084,T2:EMA_Crossover,0.6275,0.9890,0.9940,2000.0,60.0,1.0,0.0,4h,u12
9,XRPUSDT,Low,9516,12,0.897051,-0.000012,S3:Breakout_Confirm_MA,0.6060,0.8685,0.8930,2000.0,60.0,1.0,0.0,4h,u12


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,12060,12,0.0005,0.000872,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
1,BTCUSDT,Low,1.0,12060,12,0.0005,0.001982,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
2,BTCUSDT,Low,2.0,12060,12,0.0005,0.004203,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
3,BTCUSDT,Base,0.5,12060,12,0.0005,0.000878,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
4,BTCUSDT,Base,1.0,12060,12,0.0005,0.001978,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
5,BTCUSDT,Base,2.0,12060,12,0.0005,0.004196,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
6,BTCUSDT,High,0.5,12060,12,0.0005,0.000875,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
7,BTCUSDT,High,1.0,12060,12,0.0005,0.001985,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
8,BTCUSDT,High,2.0,12060,12,0.0005,0.004227,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
9,ETHUSDT,Low,0.5,12060,12,0.0005,0.001093,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12


In [ ]:
def check_expected_rows(df: pd.DataFrame, *, expected_rows: int, name: str) -> None:
    got = int(len(df))
    if got != int(expected_rows):
        raise AssertionError(f"{name}: expected {expected_rows} rows, got {got}")


def assert_universe_sizes(df: pd.DataFrame, *, expected: dict[tuple[str, str], int], name: str) -> None:
    for (symbol, cost), exp_sz in expected.items():
        g = df[(df["symbol"] == symbol) & (df["cost"] == cost)]
        if g.empty:
            raise AssertionError(f"{name}: missing ({symbol}, {cost})")
        got = int(g.iloc[0]["universe_size"])
        if got != int(exp_sz):
            raise AssertionError(f"{name}: ({symbol}, {cost}) expected universe_size={exp_sz}, got {got}")


check_expected_rows(rc_returns_1d_u17, expected_rows=15, name="rc_returns_1d_u17")
check_expected_rows(rc_returns_1d_u12, expected_rows=15, name="rc_returns_1d_u12")
check_expected_rows(rc_returns_4h_u12, expected_rows=15, name="rc_returns_4h_u12")
check_expected_rows(rc_utility_1d_u17, expected_rows=45, name="rc_utility_1d_u17")
check_expected_rows(rc_utility_1d_u12, expected_rows=45, name="rc_utility_1d_u12")
check_expected_rows(rc_utility_4h_u12, expected_rows=45, name="rc_utility_4h_u12")

expected_u12_all = {(s, c): 12 for s in SYMBOLS_1D for c in COSTS}
assert_universe_sizes(rc_returns_1d_u12, expected=expected_u12_all, name="rc_returns_1d_u12")
assert_universe_sizes(rc_returns_4h_u12, expected=expected_u12_all, name="rc_returns_4h_u12")

expected_u17_1d = {}
for s in SYMBOLS_1D:
    for c in COSTS:
        expected_u17_1d[(s, c)] = 12 if s == "BNBUSDT" else 17
assert_universe_sizes(rc_returns_1d_u17, expected=expected_u17_1d, name="rc_returns_1d_u17")

returns_all = pd.concat([rc_returns_1d_u17, rc_returns_1d_u12, rc_returns_4h_u12], ignore_index=True)
utility_all = pd.concat([rc_utility_1d_u17, rc_utility_1d_u12, rc_utility_4h_u12], ignore_index=True)

returns_view = (
    returns_all
    .groupby(["timeframe", "scenario"], as_index=False)
    .agg(
        groups=("symbol", "count"),
        rc_pvalue_median=("rc_pvalue", "median"),
        rc_pvalue_max=("rc_pvalue", "max"),
        spa_consistent_median=("spa_pvalue_consistent", "median"),
        universe_min=("universe_size", "min"),
        universe_max=("universe_size", "max"),
    )
    .sort_values(["timeframe", "scenario"])
)

utility_view = (
    utility_all
    .groupby(["timeframe", "scenario", "lam"], as_index=False)
    .agg(
        groups=("symbol", "count"),
        rc_pvalue_median=("rc_pvalue", "median"),
        spa_consistent_median=("spa_pvalue_consistent", "median"),
        universe_min=("universe_size", "min"),
        universe_max=("universe_size", "max"),
    )
    .sort_values(["timeframe", "scenario", "lam"])
)

safe_write_parquet(returns_view, RESULTS_DIR / "rc_spa_returns_compare_view.parquet")
safe_write_parquet(utility_view, RESULTS_DIR / "rc_spa_utility_compare_view.parquet")

print("Saved outputs to:", RESULTS_DIR)
display(returns_view)
display(utility_view)


Saved outputs to: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\rc_spa_tests


,timeframe,scenario,groups,rc_pvalue_median,rc_pvalue_max,spa_consistent_median,universe_min,universe_max
0,1d,u12,15,0.836582,0.963018,0.8325,12,12
1,1d,u17,15,0.840580,0.963018,0.8365,12,17
2,4h,u12,15,0.901549,0.995502,0.8700,12,12


,timeframe,scenario,lam,groups,rc_pvalue_median,spa_consistent_median,universe_min,universe_max
0,1d,u12,0.5,15,0.031484,0.0450,12,12
1,1d,u12,1.0,15,0.000500,0.0000,12,12
2,1d,u12,2.0,15,0.000500,0.0000,12,12
3,1d,u17,0.5,15,0.028486,0.0355,12,17
4,1d,u17,1.0,15,0.000500,0.0000,12,17
5,1d,u17,2.0,15,0.000500,0.0000,12,17
6,4h,u12,0.5,15,0.000500,0.0000,12,12
7,4h,u12,1.0,15,0.000500,0.0000,12,12
8,4h,u12,2.0,15,0.000500,0.0000,12,12
